# Study 2: Mitigation choices and reward costs

- Notebook 05 found that medium mitigation would have matched many high purchases on the same step. First, I compare original actions with a rule that changes every chosen high action to medium during complete episodes.
- Then I train new agents with a small reward penalty for mitigation spending and compare them with the saved original policies.
- Both comparisons use the same five training seeds and 20 held-out starts. The tables stay in memory, and the notebook does not save result files.
- It loads the current-design Q-tables from `../results/reward_comparison_current`; run Notebook 03 first. Embedded results and their interpretation are historical until refreshed.

## Settings

In [91]:
import json
import random

import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.config import TrainingConfig
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [92]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
Q_TABLE_DIRECTORY = "../results/reward_comparison_current"
CONDITIONS = ("original", "high_to_medium")

## Load the saved policies

In [ ]:
agents_by_run = {}

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        filename = f"{Q_TABLE_DIRECTORY}/q_{reward_mode}_{training_seed}.json"
        with open(filename, encoding="utf-8") as file:
            tables = json.load(file)

        agents = []
        for region, table in enumerate(tables):
            agent = TabularQLearner(region)
            for key, values in table.items():
                observation_values = []
                for value in key.split(","):
                    observation_values.append(int(value))
                observation = tuple(observation_values)
                agent.q[observation] = values
            agents.append(agent)

        agents_by_run[(reward_mode, training_seed)] = agents

print(f"Loaded {len(agents_by_run)} trained policies")


## Choose actions

- In `high_to_medium`, every chosen high action (4) becomes medium (3) before the environment step. The agent sees the resulting next observation and chooses again.

In [94]:
def choose_actions(agents, observations):
    actions = []
    for region in range(4):
        action = agents[region].select_action(observations[region], greedy=True)
        actions.append(action)
    return tuple(actions)

In [95]:
def apply_condition(chosen_actions, condition):
    if condition == "original":
        return chosen_actions

    actions = []
    for action in chosen_actions:
        actions.append(3 if action == 4 else action)
    return tuple(actions)

## Measure one episode

- GPI is averaged over the steps the episode lasts. Mitigation spending is the sum of paid low, medium, and high actions across all four regions.
- Final EDI and currency are averaged across the four regions. Peak regional load is the highest load reached by any region.
- Reward is averaged over region-steps within the same reward mode. If an episode ends early, its length and failure flag help interpret that average.

In [96]:
def empty_totals():
    return {
        "gpi": 0.0, "mitigation_spending": 0, "reward": 0.0,
        "peak_regional_load": 0.0, "development_actions": 0,
        "high_actions": 0,
        "high_replacements": 0,
    }

In [ ]:
def add_step_totals(totals, chosen_actions, actions, rewards, info):
    totals["gpi"] += info["global_pollution_index"]
    totals["reward"] += sum(rewards)
    totals["peak_regional_load"] = max(
        totals["peak_regional_load"], max(info["pollution_loads"])
    )
    totals["development_actions"] += actions.count(1)
    totals["high_actions"] += actions.count(4)
    chosen_high_actions = 0
    for action in chosen_actions:
        if action == 4:
            chosen_high_actions += 1
    totals["high_replacements"] += chosen_high_actions - actions.count(4)

    for region in range(4):
        if actions[region] in (2, 3, 4):
            totals["mitigation_spending"] += info["costs_paid"][region]


In [98]:
def episode_row(environment, totals, failed):
    region_steps = 4 * environment.step_count
    return {
        "steps": environment.step_count,
        "failure_percent": 100 * failed,
        "mean_gpi": totals["gpi"] / environment.step_count,
        "peak_regional_load": totals["peak_regional_load"],
        "mitigation_spending": totals["mitigation_spending"],
        "mean_final_edi": sum(environment.development_indices) / 4,
        "mean_final_currency": sum(environment.currency_balances) / 4,
        "development_percent": 100 * totals["development_actions"] / region_steps,
        "high_percent": 100 * totals["high_actions"] / region_steps,
        "mean_reward_per_region_step": totals["reward"] / region_steps,
        "high_replacements": totals["high_replacements"],
    }

In [99]:
def evaluate_episode(agents, reward_mode, seed, condition):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    observations = environment.reset(seed)
    totals = empty_totals()

    while True:
        chosen_actions = choose_actions(agents, observations)
        actions = apply_condition(chosen_actions, condition)
        observations, rewards, terminated, truncated, info = environment.step(actions)
        add_step_totals(totals, chosen_actions, actions, rewards, info)
        if terminated or truncated:
            break

    return episode_row(environment, totals, terminated)

## Evaluate matched starts

- `high_replacements` counts chosen high actions changed to medium during the modified episode.

In [100]:
episode_rows = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]

        for environment_seed in EVALUATION_SEEDS:
            for condition in CONDITIONS:
                row = evaluate_episode(agents, reward_mode, environment_seed, condition)
                row["reward_mode"] = reward_mode
                row["training_seed"] = training_seed
                row["environment_seed"] = environment_seed
                row["condition"] = condition
                episode_rows.append(row)

episodes = pd.DataFrame(episode_rows)
print(f"Evaluated {len(episodes)} episodes")

Evaluated 600 episodes


## Compare episode outcomes
- Lower GPI and peak load mean less pollution.
- `failure_percent` is the fraction of held-out episodes that reached the regional failure threshold.
- `mean_reward_per_region_step` is only compared between conditions **within the same reward mode**, because the modes define reward differently.

In [101]:
MEASURES = [
    "mean_gpi", "peak_regional_load", "mitigation_spending",
    "mean_final_edi", "mean_final_currency", "development_percent",
    "failure_percent", "steps", "mean_reward_per_region_step", "high_replacements",
]

seed_means = episodes.groupby(
    ["reward_mode", "training_seed", "condition"]
)[MEASURES].mean()
mode_means = seed_means.groupby(["reward_mode", "condition"]).mean()

### Pollution, spending, and final resources

In [102]:
outcome_columns = [
    "mean_gpi", "peak_regional_load", "mitigation_spending",
    "mean_final_edi", "mean_final_currency", "failure_percent", "steps",
]
display(mode_means[outcome_columns].round(1))

mean_gpi  peak_regional_load  mitigation_spending  \
reward_mode condition                                                           
global      high_to_medium       4.9                41.2              17504.4   
            original             5.0                41.0              28233.9   
local       high_to_medium       3.6                42.7              18768.2   
            original             2.6                41.5              35776.0   
mixed       high_to_medium       3.4                42.0              18760.6   
            original             2.6                41.1              35460.4   

                            mean_final_edi  mean_final_currency  \
reward_mode condition                                             
global      high_to_medium            57.1               4626.6   
            original                  58.7               2108.0   
local       high_to_medium            70.7               5545.4   
            original                  71.0               1298.3   
mixed       high_to_medium            70.0               5489.6   
            original                  68.6               1226.2   

                            failure_percent  steps  
reward_mode condition                               
global      high_to_medium              0.0  200.0  
            original                    0.0  200.0  
local       high_to_medium              0.0  200.0  
            original                    0.0  200.0  
mixed       high_to_medium              0.0  200.0  
            original                    0.0  200.0

- Changing high to medium cuts average mitigation spending from about 28,234 to 17,504 under global reward, 35,776 to 18,768 under local reward, and 35,460 to 18,761 under mixed reward.
- Mean GPI changes little for global (5.0 to 4.9), but rises for local (2.6 to 3.6) and mixed (2.6 to 3.4). I wonder how much of that increase matters when all episodes still reach 200 steps without failure and peak regional loads stay near 41-43.
- Final currency rises sharply in every mode. Final EDI changes much less: global falls by about 1.6, local by 0.4, and mixed rises by 1.4. The saved currency does not automatically turn into more development.

### Action choices and reward

- `development_percent` counts chosen development actions, whether or not they succeeded. `high_replacements` is the average number changed per episode.

In [103]:
behavior_columns = [
    "development_percent", "mean_reward_per_region_step", "high_replacements",
]
display(mode_means[behavior_columns].round(2))

development_percent  mean_reward_per_region_step  \
reward_mode condition                                                          
global      high_to_medium                 4.09                         0.02   
            original                       4.48                         0.01   
local       high_to_medium                 5.26                         0.10   
            original                       5.53                         0.16   
mixed       high_to_medium                 5.06                         0.11   
            original                       4.91                         0.18   

                            high_replacements  
reward_mode condition                          
global      high_to_medium             482.64  
            original                     0.00  
local       high_to_medium             684.40  
            original                     0.00  
mixed       high_to_medium             646.70  
            original                     0.00

- The rule changes about 483 chosen actions per global episode, 684 per local episode, and 647 per mixed episode, out of 800 region-steps. This is a large change to what the agents actually do.
- Development remains about 4-5% of attempted actions in both conditions. Replacing high with medium does little to change the tendency to stop developing.
- Mean received reward per region-step is lower under local and mixed reward after the change, by about 0.07. The global difference is small at the displayed precision. These comparisons make sense within a reward mode; the modes use different reward definitions.

## Compare each matched pair

- For each saved policy and held-out start, subtract the original outcome from the `high_to_medium` outcome.
- A negative spending difference means the modified episode spent less. A positive GPI difference means it had more pollution.
- I average paired differences within each training seed, then average the five seed results. This keeps the pairing visible without treating all 100 episodes as independent training runs.

In [104]:
pairs = episodes.pivot(
    index=["reward_mode", "training_seed", "environment_seed"],
    columns="condition", values=MEASURES,
)
changed = pairs.xs("high_to_medium", level="condition", axis=1)
original = pairs.xs("original", level="condition", axis=1)
paired_differences = changed - original

seed_differences = paired_differences.groupby(["reward_mode", "training_seed"]).mean()
mean_differences = seed_differences.groupby("reward_mode").mean()

### Differences in outcomes

In [105]:
display(mean_differences[outcome_columns].round(2))

,mean_gpi,peak_regional_load,mitigation_spending,mean_final_edi,mean_final_currency,failure_percent,steps
reward_mode,,,,,,,
global,-0.18,0.15,-10729.5,-1.60,2518.60,0.0,0.0
local,0.97,1.17,-17007.9,-0.38,4247.11,0.0,0.0
mixed,0.81,0.96,-16699.9,1.36,4263.36,0.0,0.0


- The matched differences confirm the spending reduction: about 10,730 currency per global episode and 17,000 per local or mixed episode.
- Local and mixed GPI rise by 0.97 and 0.81. Global GPI falls slightly, by 0.18, even though spending also falls. That surprised me. Changing an action changes later pollution, currency, observations, and choices, so a full-episode result need not match the one-step capacity comparison.
- Peak regional load rises by about 0.2-1.2 units, and there are no extra failures or shorter episodes. I would not call the change harmless from this alone: local and mixed policies do accept more average pollution.

### Differences in choices and reward

In [106]:
display(mean_differences[behavior_columns].round(2))

,development_percent,mean_reward_per_region_step,high_replacements
reward_mode,,,
global,-0.39,0.00,482.64
local,-0.28,-0.07,684.40
mixed,0.14,-0.07,646.70


- The small change in development choice, despite much higher final currency, keeps me interested in the reward design. More available money alone does not make these fixed Q-tables choose development often.
- Local and mixed policies receive slightly less reward with medium actions, even though they spend far less. With no direct spending penalty, the reward still favors their lower-pollution original trajectories. Does that match the balance we intended between economic growth and environmental protection?

## What if mitigation spending reduced reward?

- The earlier sections changed evaluation actions without retraining. This section trains new Q-tables with a small reward penalty for paid mitigation.
- I subtract 0.02 reward units per currency unit spent on low, medium, or high mitigation. A valid low action adds a 0.2 penalty, medium 0.5, and high 1.0. Saving and development get no new penalty.
- This is one exploratory weight, chosen to test whether directly valuing mitigation cost changes the high-action preference. It does not represent a real-world price.
- The environment's action costs, mitigation capacity, pollution updates, and failure rule stay the same. The original Q-tables remain the zero-penalty reference.

In [107]:
MITIGATION_PENALTY_WEIGHT = 0.02
TRAINING_EPISODES = 2000
training_config = TrainingConfig()

### Apply the penalty to each reward mode

- For local reward, each agent pays the penalty for its own mitigation spending.
- For global reward, each agent receives the penalty for the average spending across the four regions.
- Mixed reward uses the same 70% local and 30% global split as the original environment. Invalid attempts pay no currency, so they add no spending penalty.

In [108]:
def adjusted_rewards(rewards, info, reward_mode):
    mitigation_costs = []
    for action, cost in zip(info["attempted_actions"], info["costs_paid"]):
        mitigation_costs.append(cost if action in (2, 3, 4) else 0)

    shared_cost = sum(mitigation_costs) / 4
    adjusted = []
    for region in range(4):
        if reward_mode == "local":
            cost_for_reward = mitigation_costs[region]
        elif reward_mode == "global":
            cost_for_reward = shared_cost
        else:
            cost_for_reward = 0.7 * mitigation_costs[region] + 0.3 * shared_cost

        adjusted.append(rewards[region] - MITIGATION_PENALTY_WEIGHT * cost_for_reward)

    return tuple(adjusted)

### Train with the new reward

- Each reward mode uses the same five training seeds, 2,000 episodes, Q-learning settings, and episode seeds as notebook 03

In [109]:
def choose_actions_for_training(agents, observations):
    actions = []
    for region in range(4):
        actions.append(agents[region].select_action(observations[region]))
    return tuple(actions)

In [110]:
def train_penalty_episode(environment, agents, seed):
    observations = environment.reset(seed)

    while True:
        actions = choose_actions_for_training(agents, observations)
        next_observations, rewards, terminated, truncated, info = environment.step(actions)
        rewards = adjusted_rewards(rewards, info, environment.reward_mode)

        for region in range(4):
            agents[region].update(
                observations[region], actions[region], rewards[region],
                next_observations[region], terminated,
            )

        observations = next_observations
        if terminated or truncated:
            break

In [111]:
penalty_agents = {}

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        environment = EconomicSustainabilityEnvironment(reward_mode)
        episode_rng = random.Random(training_seed)
        agents = []
        for region in range(4):
            agents.append(TabularQLearner(training_seed * 10 + region, training_config))

        for episode in range(TRAINING_EPISODES):
            seed = episode_rng.randrange(100_000)
            train_penalty_episode(environment, agents, seed)
            for agent in agents:
                agent.decay_epsilon()

        penalty_agents[(reward_mode, training_seed)] = agents
        print(f"Trained {reward_mode}, seed {training_seed}")

Trained local, seed 11
Trained local, seed 23
Trained local, seed 37
Trained local, seed 41
Trained local, seed 53
Trained global, seed 11
Trained global, seed 23
Trained global, seed 37
Trained global, seed 41
Trained global, seed 53
Trained mixed, seed 11
Trained mixed, seed 23
Trained mixed, seed 37
Trained mixed, seed 41
Trained mixed, seed 53


### Evaluate the newly trained policies

In [112]:
penalty_rows = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = penalty_agents[(reward_mode, training_seed)]
        for environment_seed in EVALUATION_SEEDS:
            row = evaluate_episode(agents, reward_mode, environment_seed, "original")
            row["reward_mode"] = reward_mode
            row["training_seed"] = training_seed
            row["environment_seed"] = environment_seed
            penalty_rows.append(row)

penalty_episodes = pd.DataFrame(penalty_rows)
print(f"Evaluated {len(penalty_episodes)} penalty-trained episodes")

Evaluated 300 penalty-trained episodes


In [113]:
reference_episodes = episodes[episodes["condition"] == "original"].copy()
reference_episodes["reward_setting"] = "original"
penalty_episodes["reward_setting"] = "mitigation_penalty"
comparison = pd.concat([reference_episodes, penalty_episodes], ignore_index=True)

### Compare outcomes

In [120]:
PENALTY_MEASURES = [
    "mean_gpi", "peak_regional_load", "mitigation_spending",
    "mean_final_edi", "mean_final_currency", "failure_percent",
    "steps", "development_percent", "high_percent",
    "mean_reward_per_region_step",
]

penalty_by_seed = comparison.groupby(
    ["reward_mode", "training_seed", "reward_setting"]
)[PENALTY_MEASURES].mean()
penalty_summary = penalty_by_seed.groupby(["reward_mode", "reward_setting"]).mean()

In [115]:
penalty_outcomes = [
    "mean_gpi", "peak_regional_load", "mitigation_spending",
    "mean_final_edi", "mean_final_currency", "failure_percent", "steps",
]
display(penalty_summary[penalty_outcomes].round(1))

mean_gpi  peak_regional_load  \
reward_mode reward_setting                                     
global      mitigation_penalty       3.9                40.7   
            original                 5.0                41.0   
local       mitigation_penalty       2.7                41.3   
            original                 2.6                41.5   
mixed       mitigation_penalty       2.9                41.1   
            original                 2.6                41.1   

                                mitigation_spending  mean_final_edi  \
reward_mode reward_setting                                            
global      mitigation_penalty              25943.2            62.2   
            original                        28233.9            58.7   
local       mitigation_penalty              18924.2            68.0   
            original                        35776.0            71.0   
mixed       mitigation_penalty              18961.2            69.0   
            original                        35460.4            68.6   

                                mean_final_currency  failure_percent  steps  
reward_mode reward_setting                                                   
global      mitigation_penalty               3057.2              0.0  200.0  
            original                         2108.0              0.0  200.0  
local       mitigation_penalty               5324.3              0.0  200.0  
            original                         1298.3              0.0  200.0  
mixed       mitigation_penalty               5363.4              0.0  200.0  
            original                         1226.2              0.0  200.0

- Local and mixed policies spend much less with the penalty: about 18,900 currency per episode instead of 35,000-36,000. Their mean GPI rises only a little, from 2.6 to 2.7 for local and 2.6 to 2.9 for mixed. Every held-out episode still lasts 200 steps without failure.
- Global spending falls less, from about 28,234 to 25,943. Its mean GPI falls from 5.0 to 3.9, and mean final EDI rises from 58.7 to 62.2. I would check variation across seeds before reading that as a dependable improvement.
- Final currency rises in every mode. Local final EDI falls from 71.0 to 68.0, while mixed stays close to 69.0. Saving money is still not consistently turning into more development.

### Compare actions and the original reward

- `development_percent` and `high_percent` count attempted actions. `mean_reward_per_region_step` uses the original reward for both conditions.

In [116]:
penalty_behavior = [
    "development_percent", "high_percent", "mean_reward_per_region_step",
]
display(penalty_summary[penalty_behavior].round(2))

development_percent  high_percent  \
reward_mode reward_setting                                          
global      mitigation_penalty                 4.29         41.53   
            original                           4.48         57.18   
local       mitigation_penalty                 4.86          1.09   
            original                           5.53         85.54   
mixed       mitigation_penalty                 4.92          2.22   
            original                           4.91         84.33   

                                mean_reward_per_region_step  
reward_mode reward_setting                                   
global      mitigation_penalty                        -0.01  
            original                                   0.01  
local       mitigation_penalty                         0.18  
            original                                   0.16  
mixed       mitigation_penalty                         0.13  
            original                                   0.18

- The action change is striking under local and mixed reward. High mitigation falls from 85.54% to 1.09% of local region-steps and from 84.33% to 2.22% of mixed region-steps. Global falls from 57.18% to 41.53%.
- Development choices barely change: around 4-5% of region-steps in every condition. This penalty discourages high mitigation, but it does not by itself create a stronger development policy.
- The last column scores both policies with the original reward during evaluation. It is not the penalty-adjusted reward used to train the new agents, so I would not use it alone to rank these reward designs.

### Compare matched differences across seeds

- Subtract the original policy's episode outcome from the penalty-trained policy's outcome on the same start.
- The tables show the mean change across five training seeds and its standard deviation (`std`). Five seeds are an initial comparison, so variation matters.

In [117]:
matched = comparison.pivot(
    index=["reward_mode", "training_seed", "environment_seed"],
    columns="reward_setting", values=PENALTY_MEASURES,
)
penalty_values = matched.xs("mitigation_penalty", level="reward_setting", axis=1)
original_values = matched.xs("original", level="reward_setting", axis=1)
episode_changes = penalty_values - original_values
seed_changes = episode_changes.groupby(["reward_mode", "training_seed"]).mean()

#### Pollution, spending, development, and failure

In [118]:
outcome_changes = [
    "mean_gpi", "mitigation_spending", "mean_final_edi", "failure_percent",
]
outcome_change_summary = seed_changes.groupby("reward_mode")[outcome_changes].agg(["mean", "std"])
display(outcome_change_summary.round(2))

mean_gpi       mitigation_spending          mean_final_edi         \
                mean   std                mean      std           mean    std   
reward_mode                                                                     
global         -1.16  3.02            -2290.75  7517.78           3.58  14.62   
local           0.11  0.44           -16851.80  1217.37          -3.10   4.60   
mixed           0.31  0.12           -16499.25  1086.75           0.34   3.86   

            failure_percent       
                       mean  std  
reward_mode                       
global                  0.0  0.0  
local                   0.0  0.0  
mixed                   0.0  0.0

- Local and mixed spending reductions are fairly consistent across the five training seeds: about 16,852 and 16,499 currency per episode, with standard deviations around 1,100-1,200. Their GPI increases are much smaller, 0.11 and 0.31.
- The global averages are harder to interpret. Its mean GPI change is -1.16, but the standard deviation across seeds is 3.02. The spending change is -2,291 with a standard deviation of 7,518, and final EDI also varies widely. I would inspect the individual global seeds before claiming the penalty helps that mode.
- Failure remains at 0% in both conditions on these starts. That does not tell us which pollution-spending balance is preferable.

#### Development and high mitigation choices

In [119]:
action_changes = ["development_percent", "high_percent"]
action_change_summary = seed_changes.groupby("reward_mode")[action_changes].agg(["mean", "std"])
display(action_change_summary.round(2))

development_percent       high_percent       
                           mean   std         mean    std
reward_mode                                              
global                    -0.20  0.91       -15.65  31.53
local                     -0.67  0.87       -84.44   6.09
mixed                      0.01  0.39       -82.11   5.25

- The penalty reduces high mitigation by about 84 percentage points under local reward and 82 under mixed reward, with much less seed variation than global. Global falls by about 16 points on average, but its standard deviation is 32 points.
- Development choice changes by less than one percentage point in every mode. This makes me think the missing spending penalty explains much of the high-mitigation preference for local and mixed agents, while the weak development incentive needs its own investigation.
- I wonder whether the shared reward helps explain the variable global result. One agent's mitigation penalty is averaged across four regions in global mode, so its own high purchase has a smaller direct effect on the reward it receives.